In [2]:
!pip install groq --quiet
import os
import json
import re
import time
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.8 MB/s eta 0:00:00


In [3]:
from groq import Groq
API_KEY = "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"
client = Groq(api_key=API_KEY)
MODEL = "llama-3.1-8b-instant"
print("Groq client configured with model:",MODEL)
print("Make sure API_KEY is replaced with your actual key!")


Groq client configured with model: llama-3.1-8b-instant
Make sure API_KEY is replaced with your actual key!


In [4]:
print("==== FIRST LLM API CALL ====")
print()

def ask_llm(user_message,system_message="you are a helpful assistant",
            temperature=0.7,max_tokens=500):
    """
    send a message to the LLM and return the response text

    """

    response = client.chat.completions.create(
        model = MODEL,
        messages = [
            {"role":"system","content":system_message},
            {"role":"user","content":user_message}
        ],
        temperature = temperature,
        max_tokens = max_tokens,

    )
    return response.choices[0].message.content

test_response = ask_llm(
    "What is AI"
)
print("=== LLM Response===")
print(test_response)

==== FIRST LLM API CALL ====

=== LLM Response===
AI stands for Artificial Intelligence, which refers to the simulation of human intelligence in machines that are programmed to think and learn like humans. The term may also be applied to any machine that exhibits traits associated with a human mind such as learning and problem-solving.

AI technology is based on the principle of creating algorithms that can process data and make decisions, often with minimal human intervention. These algorithms can be trained on large datasets to enable the AI system to learn from experience and improve its performance over time.

There are several types of AI, including:

1. **Narrow or Weak AI**: This type of AI is designed to perform a specific task, such as playing chess, recognizing faces, or translating languages.
2. **General or Strong AI**: This type of AI is designed to perform any intellectual task that a human can, such as reasoning, problem-solving, and learning.
3. **Superintelligence**: T

In [5]:
response_etl = ask_llm(
    "In 3 bullet points,  explain how the medallion architecture"
    "(bronze,silver,gold layers) relates to the ETL pipeline",
    system_message="you are a senior data engineering instructor. "
                    "Be concise and practical"

)
print("Medallion + ETL Connection:")
print(response_etl)
print()
print("=== Token Explanation ===")
print("Each word is roughly 1-2 tokens:")
print("The model above used approximately",len(response_etl.split()),"tokens")
print("llama-3.1-8b context window: 8192 tokens(6000 words per conversation)")

Medallion + ETL Connection:
Here are three bullet points explaining the Medallion Architecture and its relation to the ETL pipeline:

• **Bronze Layer (Raw Data Landing Zone)**: This layer represents the Ingestion phase of the ETL pipeline, where raw data from various sources (e.g., databases, APIs, files) is collected, stored, and made available for processing. The Bronze layer acts as a data lake, where data is stored in its native format, making it a central hub for data integration and processing.

• **Silver Layer (Data Processing and Transformation)**: This layer represents the Transformation and Load phases of the ETL pipeline, where data from the Bronze layer is processed, transformed, and aggregated into a more usable format. The Silver layer is where data quality checks, data cleansing, and data transformations occur, making it easier to work with the data for analysis and reporting.

• **Gold Layer (Data Marts and Analytical Data)**: This layer represents the final output of

In [6]:
print("=== Zero Shot Prompting ===")
print()

# Zero-shot prompting using LLM
zero_shot_response = ask_llm(
    "Extract the city name from this address: "
    "456 Brigade Road, Bangalore 500025, Karnataka, India"
)

print("Zero shot results:")
print(zero_shot_response)
print()

# Ambiguous prompt example
ambiguous_response = ask_llm(
    "Clean this data: ramesh kumar,45000,mumbai"
)

print("Ambiguous prompt results:")
print(ambiguous_response)
print()

print("Problem: output format is unpredictable and not machine-parseable!")

=== Zero Shot Prompting ===

Zero shot results:
The city name extracted from the address is: Bangalore.

Ambiguous prompt results:
Based on the given data "ramesh kumar,45000,mumbai", I'll make an assumption that the data is in the format of "name,salary,location". Here's the cleaned data:

- Name: Ramesh Kumar
- Salary: 45,000
- Location: Mumbai

If the data is meant to be in a table format, it might look like this:

| Name          | Salary    | Location |
|---------------|-----------|----------|
| Ramesh Kumar  | 45,000    | Mumbai   |

Problem: output format is unpredictable and not machine-parseable!


In [7]:
import json

few_shot_prompt = """
Convert employee text into JSON. Here are the examples.


Input: Ravi,50000,Chennai
Output: {"name":"Ravi","salary":"50000","city":"Chennai"}

Input: Priya,60000,Bangalore
Output: {"name":"Priya","salary":"60000","city":"Bangalore"}

Input: Ramesh,45000,Mumbai
Output:
Return ONLY valid JSON.
"""

few_shot_response = ask_llm(
    few_shot_prompt,
    temperature=0.0
)

print("Few Shot Results:")
print(few_shot_response)

try:
    parsed = json.loads(few_shot_response.strip())

    print("Successfully parsed as JSON!")

    print(
        "Name:", parsed["name"],
        "Salary:", parsed["salary"],
        "City:", parsed["city"]
    )

except json.JSONDecodeError:
    print("Parsing failed - model added extra text")
    print("Solution: add explicit instructions in the prompt")

Few Shot Results:
Here's a Python function that converts the employee text into JSON:

```python
import json

def convert_to_json(employee_text):
    """
    Converts employee text into JSON.

    Args:
        employee_text (str): Employee details in the format "name,salary,city".

    Returns:
        str: Valid JSON string.
    """
    try:
        name, salary, city = employee_text.split(',')
        return json.dumps({"name": name, "salary": salary, "city": city})
    except ValueError:
        return None

# Test the function
print(convert_to_json("Ravi,50000,Chennai"))  # Output: {"name": "Ravi", "salary": "50000", "city": "Chennai"}
print(convert_to_json("Priya,60000,Bangalore"))  # Output: {"name": "Priya", "salary": "60000", "city": "Bangalore"}
print(convert_to_json("Ramesh,45000,Mumbai"))  # Output: {"name": "Ramesh", "salary": "45000", "city": "Mumbai"}
print(convert_to_json("Invalid employee text"))  # Output: None
```

This function uses the `json.dumps()` method to conv

In [8]:
same_question = "Review this python code and identify any issues:"\
                "df['revenue'] = df['city']"
generic_response = ask_llm(
    same_question,temperature=0.2
)
print("without Role prompting")
print(generic_response[:300], '....')
print()

role_response = ask_llm(
    same_question,
    system_message = "you are a senior data engineer with 10 years of production"
                      "experience. review the code critically for production readiness,"
                      "data type issues, and potential failures at scale",
    temperature = 0.2
  )
print("with role prompting(senior data engineer):")
print(role_response[:400],'....')
print()

without Role prompting
The provided Python code is assigning the values of the 'city' column to the 'revenue' column in a pandas DataFrame 'df'. This is likely not the intended behavior, as it will result in the 'revenue' column containing city names instead of revenue values.

Here are a few potential issues with this co ....

with role prompting(senior data engineer):
**Code Review**

The provided Python code is assigning the values of the 'city' column to the 'revenue' column in a pandas DataFrame. However, this is likely to cause issues due to the mismatch in data types and potential data loss.

```python
df['revenue'] = df['city']
```

**Issues:**

1. **Data Type Mismatch**: The 'city' column likely contains string values, while the 'revenue' column is expec ....



In [9]:
# Temperature = creativeness of the model we trained
import time
prompt = "Give me one creative for a data analytics startup"

print("=== Temperature Experiment ===")
print()
for temp in [0.0,0.5,1.0]:
  response = ask_llm(prompt,temperature = temp)
  print(f"Temperature  {temp} : {response.strip()}")
  time.sleep(1)
print()
print("Observation")
print(" Temperature = 0.0 -> same or very similar answer every run (deterministic)")
print(" Temperature = 0.5 -> some variation")
print(" Temperature = 1.0 -> very creative and surprising")
print()
print("Rule for data engieering task : use temperature = 0.0 or 0.1")
print("You need CONSISTENT , PARSEABLE output - not creative variation")

=== Temperature Experiment ===

Temperature  0.0 : Here's a creative concept for a data analytics startup:

**Company Name:** Lumina Insights

**Tagline:** "Illuminate Your Business with Data"

**Concept:** Lumina Insights is a cutting-edge data analytics startup that uses AI-powered predictive modeling to help businesses make data-driven decisions. The company's mission is to empower organizations to navigate uncertainty and drive growth by providing actionable insights from complex data sets.

**Unique Selling Proposition (USP):** Lumina Insights' proprietary platform, called "Lumina Lens," uses machine learning algorithms to analyze vast amounts of data from various sources, including customer behavior, market trends, and operational metrics. The platform provides real-time insights and predictive models that enable businesses to anticipate and respond to changes in their market, optimize their operations, and make informed decisions.

**Key Features:**

1. **Predictive Modeling:** 

In [11]:
messy_invoices = [
    "INV-2024-0891 TECHWORLD SOLUTIONS 15th jan 2024 Rs. 45,000 Laptop purchase",
    "Invoice from PRIYA ENTERPRISES dt 07-02-2024 amt: 12500 for office cleaning services",
    "#INV-2024-103 | arjun nair consultancy | 8000 | march 15 2024 | python training",
    "SURESH AND HARDWARE STORE 2500 keyboard and mouse accessories 2024/01/20",
    "Tax Invoice: Ananya tech solutions | INV-897 | Date: 28-Feb-24 | Amount: INR 95,000| Server Hardware"
]

print("Messy Invoices:")
for i,inv in enumerate(messy_invoices,1):
  print(f"{i}. {inv}")
print("total:",len(messy_invoices))

Messy Invoices:
1. INV-2024-0891 TECHWORLD SOLUTIONS 15th jan 2024 Rs. 45,000 Laptop purchase
2. Invoice from PRIYA ENTERPRISES dt 07-02-2024 amt: 12500 for office cleaning services
3. #INV-2024-103 | arjun nair consultancy | 8000 | march 15 2024 | python training
4. SURESH AND HARDWARE STORE 2500 keyboard and mouse accessories 2024/01/20
5. Tax Invoice: Ananya tech solutions | INV-897 | Date: 28-Feb-24 | Amount: INR 95,000| Server Hardware
total: 5


In [12]:
few_shot_prompt = """
Extract employee name and salary from the text and return JSON format.

Example 1:
Input: Ravi earns 50000 rupees per month.
Output: {"name":"Ravi","salary":"50000"}

Example 2:
Input: Priya salary is 65000 in Bangalore office.
Output: {"name":"Priya","salary":"65000"}

Example 3:
Input: Arjun gets 72000 monthly package.
Output: {"name":"Arjun","salary":"72000"}

Now extract details from this text:

Input: Ramesh earns 45000 rupees in Mumbai.
Output:
"""

few_shot_response = ask_llm(
    few_shot_prompt,
    system_message="""
    You are a data extraction assistant.

    Rules:
    - Return ONLY valid JSON
    - No explanations
    - No markdown
    """,
    temperature=0.0
)

print("Few Shot Result:")
print(few_shot_response)

Few Shot Result:
{"name":"Ramesh","salary":"45000"}


In [15]:
extracted_records = [
    {
        "invoice_id": "INV101",
        "customer": "Ravi Stores",
        "amount": "4500",
        "invoice_date": "2026-08-01"
    },
    {
        "invoice_id": "INV102",
        "customer": "Priya Traders",
        "amount": "7200",
        "invoice_date": "2026-08-02"
    },
    {
        "invoice_id": "INV103",
        "customer": "Arjun Mart",
        "amount": "5600",
        "invoice_date": "2026-08-03"
    }
]

In [19]:
# Convert list of dicts to Pandas DataFrame (Day 3 skill)
invoices_df = pd.DataFrame(extracted_records)

# Clean up the DataFrame
invoices_df['amount'] = pd.to_numeric(
    invoices_df['amount'],
    errors='coerce'
)

# Convert date column
invoices_df['invoice_date'] = pd.to_datetime(
    invoices_df['invoice_date'],
    errors='coerce'
)

print("=== SMART DATA CLEANER OUTPUT ===")
print(f"Rows: {len(invoices_df)} | Columns: {len(invoices_df.columns)}")
print()

print(invoices_df.to_string(index=False))

=== SMART DATA CLEANER OUTPUT ===
Rows: 3 | Columns: 4

invoice_id      customer  amount invoice_date
    INV101   Ravi Stores    4500   2026-08-01
    INV102 Priya Traders    7200   2026-08-02
    INV103    Arjun Mart    5600   2026-08-03
